In [3]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

id_digits = ''.join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, 'Student ID must contain digits.'

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print('Student:', LAST_NAME, STUDENT_ID)
print('Seed Digit:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)
print('Random Seed:', RANDOM_SEED)
print('Verification Value:', verification_value)

Student: Magalong TUPM-26-1134
Seed Digit: 4
ID Sum: 17
Name Length: 8
Random Seed: 261134
Verification Value: 140


In [4]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / 'secom.data'
label_path = DATA_DIR / 'secom_labels.data'
description_path = DATA_DIR / 'secom.names'

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f'Missing file: {path}'

print('Workspace:', WORK_DIR.resolve())
print('All dataset files verified.')

Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab7_NumPy_Pandas_Magalong\Lab7_Additionals
All dataset files verified.


In [5]:
# [CELL 3] LOAD MEASUREMENTS AND IDENTIFY UNUSABLE FEATURES
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

features_with_missing = missing.any(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

print('Shape:', raw_measurements.shape)
print('Total Missing Values:', missing.sum())
print('Features With Missing Values:', features_with_missing.sum())
print('Entirely Missing Features:', all_missing.sum())
print('Constant Features:', constant.sum())
print('Usable Features:', len(usable_indices))

Shape: (1567, 590)
Total Missing Values: 41951
Features With Missing Values: 538
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [6]:
# [CELL 4] SELECT USABLE FEATURES REPRODUCIBLY
n_select = 5 + (SEED_DIGIT % 3)

rng = np.random.default_rng(RANDOM_SEED)
positions = rng.choice(len(usable_indices), size=n_select, replace=False)

selected_indices = usable_indices[positions]
selected_names = [f'Feature_{i:03d}' for i in selected_indices]

X = raw_measurements[:, selected_indices]

print('Features to Select:', n_select)
print('Selected Features:', selected_names)
print('Selected Matrix Shape:', X.shape)

Features to Select: 6
Selected Features: ['Feature_214', 'Feature_344', 'Feature_523', 'Feature_029', 'Feature_286', 'Feature_405']
Selected Matrix Shape: (1567, 6)


In [7]:
# [CELL 5] FEATURE-WISE STATISTICS USING NUMPY
selected_missing = np.isnan(X)
observed_counts = (~selected_missing).sum(axis=0)

feature_means = np.nanmean(X, axis=0)
feature_medians = np.nanmedian(X, axis=0)
feature_sds = np.nanstd(X, axis=0, ddof=1)
missing_pct = 100 * selected_missing.mean(axis=0)

for i, name in enumerate(selected_names):
    print(f'{name}: mean={feature_means[i]:.4f}, median={feature_medians[i]:.4f}, '
          f'sd={feature_sds[i]:.4f}, missing={missing_pct[i]:.2f}%')

print('Highest Missing Percentage:', f'{missing_pct.max():.2f}%',
      '-', selected_names[int(np.argmax(missing_pct))])

Feature_214: mean=0.0833, median=0.0754, sd=0.0565, missing=1.53%
Feature_344: mean=23.2171, median=23.2147, sd=8.8952, missing=0.38%
Feature_523: mean=0.4539, median=0.1000, sd=4.1476, missing=0.00%
Feature_029: mean=2.3662, median=2.3778, sd=0.4087, missing=0.13%
Feature_286: mean=4.2600, median=3.3601, sd=9.7638, missing=0.19%
Feature_405: mean=0.0242, median=0.0239, sd=0.0107, missing=0.51%
Highest Missing Percentage: 1.53% - Feature_214


In [8]:
# [CELL 6] ROBUST STANDARDIZATION USING MEDIAN AND MAD
THRESHOLD = 3.5

abs_deviation = np.abs(X - feature_medians)
mad = np.nanmedian(abs_deviation, axis=0)

zero_mad = mad == 0
safe_mad = np.where(zero_mad, 1.0, mad)

robust_z = 0.6745 * (X - feature_medians) / safe_mad

# A zero MAD gives no usable scale: observed values get 0.0, missing stay NaN
robust_z = np.where(zero_mad, np.where(selected_missing, np.nan, 0.0), robust_z)

flags = np.abs(robust_z) > THRESHOLD

flagged_counts = flags.sum(axis=0)
flagged_pct = 100 * flagged_counts / observed_counts

print('Threshold:', THRESHOLD)
print('Zero-MAD Features:', zero_mad.sum())
print('Flagged Counts:', flagged_counts)
print('Flagged Percentages:', flagged_pct)
print('Total Flagged Observations:', flags.sum())

Threshold: 3.5
Zero-MAD Features: 0
Flagged Counts: [ 53   7 192   5  21   0]
Flagged Percentages: [ 3.4349  0.4484 12.2527  0.3195  1.3427  0.    ]
Total Flagged Observations: 278


In [9]:
# [CELL 7] RANK FEATURES AND EXPORT CSV SUMMARY
order = np.argsort(-flagged_pct, kind='stable')

quality = pd.DataFrame({
    'Rank': np.arange(1, n_select + 1),
    'Feature': [selected_names[i] for i in order],
    'Observed_Count': observed_counts[order],
    'Missing_Percentage': missing_pct[order],
    'Mean': feature_means[order],
    'Median': feature_medians[order],
    'Sample_SD': feature_sds[order],
    'MAD': mad[order],
    'Zero_MAD': zero_mad[order],
    'Flagged_Count': flagged_counts[order],
    'Flagged_Percentage': flagged_pct[order]
})

quality_path = OUTPUT_DIR / 'measurement_quality.csv'
quality.to_csv(quality_path, index=False)

print(quality.round(4))
print('Created:', quality_path.name, quality_path.is_file())

   Rank      Feature  Observed_Count  Missing_Percentage     Mean   Median  \
0     1  Feature_523            1567              0.0000   0.4539   0.1000   
1     2  Feature_214            1543              1.5316   0.0833   0.0754   
2     3  Feature_286            1564              0.1914   4.2600   3.3600   
3     4  Feature_344            1561              0.3829  23.2171  23.2147   
4     5  Feature_029            1565              0.1276   2.3662   2.3778   
5     6  Feature_405            1559              0.5105   0.0242   0.0239   

   Sample_SD     MAD  Zero_MAD  Flagged_Count  Flagged_Percentage  
0     4.1476  0.0288     False            192             12.2527  
1     0.0565  0.0185     False             53              3.4349  
2     9.7638  0.9200     False             21              1.3427  
3     8.8952  5.4547     False              7              0.4484  
4     0.4087  0.2889     False              5              0.3195  
5     0.0107  0.0093     False              0

In [10]:
# [CELL 8] VERIFY VECTORIZED RESULTS
# Independent loop-based check of the flag counts
loop_counts = []
for j in range(n_select):
    count = 0
    for i in range(X.shape[0]):
        value = X[i, j]
        if value != value or mad[j] == 0:
            continue
        if abs(0.6745 * (value - feature_medians[j]) / mad[j]) > THRESHOLD:
            count += 1
    loop_counts.append(count)

counts_match = np.array_equal(loop_counts, flagged_counts)

# Cross-check descriptive statistics against Pandas
check = pd.DataFrame(X)
stats_match = (
    np.allclose(check.mean().to_numpy(), feature_means)
    and np.allclose(check.median().to_numpy(), feature_medians)
    and np.allclose(check.std(ddof=1).to_numpy(), feature_sds)
)

loaded = pd.read_csv(quality_path)
export_match = (len(loaded) == n_select
                and loaded['Flagged_Count'].sum() == flags.sum())

vectorized_verified = counts_match and stats_match and export_match

print('Loop Flag Counts:', loop_counts)
print('Flag Counts Match:', counts_match)
print('NumPy-Pandas Statistics Match:', stats_match)
print('Exported File Matches:', export_match)
print('Vectorized Results Verified:', vectorized_verified)

Loop Flag Counts: [53, 7, 192, 5, 21, 0]
Flag Counts Match: True
NumPy-Pandas Statistics Match: True
Exported File Matches: True
Vectorized Results Verified: True


In [11]:
# [CELL 9] FINAL OUTPUT
top = int(order[0])

print('ASSESSMENT DATA - VECTORIZED MEASUREMENT QUALITY ANALYSIS')
print('---------------------------------------------------------')
print('Selected Features:', ', '.join(selected_names))
print('Highest Missing Percentage:', f'{missing_pct.max():.2f}%')
print('Feature with Highest Flagged Percentage:',
      selected_names[top], f'({flagged_pct[top]:.2f}%)')
print('Total Flagged Observations:', int(flags.sum()))
print('Vectorized Results Verified:', vectorized_verified)
print('Required Output:', quality_path.name)

ASSESSMENT DATA - VECTORIZED MEASUREMENT QUALITY ANALYSIS
---------------------------------------------------------
Selected Features: Feature_214, Feature_344, Feature_523, Feature_029, Feature_286, Feature_405
Highest Missing Percentage: 1.53%
Feature with Highest Flagged Percentage: Feature_523 (12.25%)
Total Flagged Observations: 278
Vectorized Results Verified: True
Required Output: measurement_quality.csv
